In [14]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from fundus_data_toolkit.datamodules.classification import DDRDataModule
from multistyleseg.experiments.fundus.ensemble import get_ensemble_model
from multistyleseg.data.fundus.consts import Lesions, ALL_CLASSES
import torch
from tqdm.auto import tqdm
from multiprocessing import Pool
import cv2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
ddr_datamodule = DDRDataModule(
    data_dir="/home/clement/Documents/data/DDR-dataset/DR_grading/",
    img_size=1024,
    batch_size=16,
    num_workers=4,
)
ddr_datamodule = ddr_datamodule.setup_all()

In [16]:
ddr_datamodule.train.return_indices = True
dataloader = ddr_datamodule.train_dataloader()

In [17]:
ensemble = get_ensemble_model(model_choices=["UNET", "SEGFORMER", "SERESNET UNET"])
ensemble.eval()
ensemble = ensemble.cuda()

In [18]:
from skimage.measure import label, regionprops, find_contours
import pandas as pd
import numpy as np

outputs_folders = Path("train/ensemble_inference")
output_segmentations = outputs_folders / "segmentations"
output_lesions_folders = outputs_folders / "lesions"
outputs_folders.mkdir(exist_ok=True, parents=True)
output_lesions_folders.mkdir(exist_ok=True, parents=True)
output_segmentations.mkdir(exist_ok=True, parents=True)

dataset = dataloader.dataset
root_img = Path(dataloader.dataset.img_root[0])
for batch in tqdm(dataloader, total=len(dataloader)):
    indices = batch["index"]
    batch_done = False
    for index in indices:
        # Check if output already exists, skip if so
        filepath = Path(dataset.img_filepath["image"][index])
        relative_path = filepath.relative_to(root_img)
        output_lesion = output_segmentations / f"{relative_path.stem}.png"
        if output_lesion.exists():
            batch_done = True
            break
    if batch_done:
        continue
    images = batch["image"].cuda()

    with torch.inference_mode():
        with torch.no_grad():
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = ensemble(images)
                predictions = torch.argmax(outputs, dim=1).cpu().numpy()
    for i, index in enumerate(indices):
        filepath = Path(dataset.img_filepath["image"][index])
        relative_path = filepath.relative_to(root_img)
        output_lesion = output_segmentations / f"{relative_path.stem}.png"
        output_lesion.parent.mkdir(exist_ok=True, parents=True)
        prediction = predictions[i]
        cv2.imwrite(str(output_lesion), prediction.astype(np.uint8))

  0%|          | 0/783 [00:00<?, ?it/s]

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)


In [19]:
def extract_lesions(segmentation_map_filepath):
    prediction = cv2.imread(str(segmentation_map_filepath), cv2.IMREAD_UNCHANGED)
    if prediction is None:
        return

    imageId = segmentation_map_filepath.stem

    components = label(prediction)
    props = regionprops(components, intensity_image=prediction)

    lesions_id = []
    lesions_contours = []
    for prop in props:
        class_idx = int(prop.max_intensity) - 1
        if 0 <= class_idx < len(ALL_CLASSES):
            lesions_id.append(ALL_CLASSES[class_idx].name)
        else:
            lesions_id.append("UNKNOWN")

        lesion_mask = (components == prop.label).astype(np.uint8)
        contours = find_contours(lesion_mask, 0.5)
        lesions_contours.append(contours)

    data = dict(
        image_id=[imageId] * len(props),
        lesion_id=lesions_id,
        image_path=[str(segmentation_map_filepath)] * len(props),
        area=[prop.area for prop in props],
        centroid=[prop.centroid for prop in props],
        bbox=[prop.bbox for prop in props],
        coordinates=[prop.coords for prop in props],
        contours=lesions_contours,
    )
    df = pd.DataFrame(data)
    output_pkl = (output_lesions_folders / segmentation_map_filepath.stem).with_suffix(
        ".pkl"
    )
    df.to_pickle(output_pkl)


segmentation_files = list(output_segmentations.glob("*.png"))

with Pool() as pool:
    list(
        tqdm(
            pool.imap(extract_lesions, segmentation_files),
            total=len(segmentation_files),
        )
    )

  0%|          | 0/6260 [00:00<?, ?it/s]